# 05 - Attrition Model

I compare logistic regression against a random forest. Accuracy is a trap here: the do-nothing baseline is ~84%.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, recall_score, accuracy_score

hr = pd.read_csv('data/processed/hr_cleaned.csv')
features = hr.drop(columns=['Attrition', 'AttritionFlag'])
target = hr['AttritionFlag']

## Encode categorical columns

Ordinals stay numeric; nominals become dummy variables.

In [ ]:
categorical_columns = ['BusinessTravel','Department','EducationField','Gender','JobRole','MaritalStatus','OverTime','AgeBand','TenureBand','DistanceBand','IncomeBand']
features = pd.get_dummies(features, columns=categorical_columns, drop_first=True)
features.shape

## Stratified split and models

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=42, stratify=target)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
log_model.fit(X_train_scaled, y_train)
log_prob = log_model.predict_proba(X_test_scaled)[:, 1]
print('Baseline accuracy (always predict No): %.3f' % (1 - y_test.mean()))
print('Logistic  accuracy=%.3f recall=%.3f roc_auc=%.3f' % (accuracy_score(y_test, log_model.predict(X_test_scaled)), recall_score(y_test, log_model.predict(X_test_scaled)), roc_auc_score(y_test, log_prob)))

In [ ]:
rf_model = RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
rf_prob = rf_model.predict_proba(X_test)[:, 1]
print('RandomForest accuracy=%.3f recall=%.3f roc_auc=%.3f' % (accuracy_score(y_test, rf_model.predict(X_test)), recall_score(y_test, rf_model.predict(X_test)), roc_auc_score(y_test, rf_prob)))

## Interpretation: odds ratios and importances

In [ ]:
coef_table = pd.DataFrame({'Feature': features.columns, 'OddsRatio': np.exp(log_model.coef_[0])}).sort_values('OddsRatio', ascending=False)
import numpy as np
coef_table.head(10)